In [ ]:
# ============================================================
# CONFIG  & IMPORTS
# ============================================================
import re
import json
import time
import itertools
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import emoji

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
from torch.cuda.amp import GradScaler, autocast          # ← AMP

import nltk
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('vader_lexicon', quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── GPU speed-up flags ──────────────────────────────────────
torch.backends.cudnn.benchmark = True                     # auto-tune kernel CNN/LSTM
torch.set_float32_matmul_precision('high')                # aktifkan TF32 (Ampere/T4)
torch.cuda.empty_cache()

LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

experiment_results = []

# ── Helper: banner & print terstandar ───────────────────────
def print_banner(title, device=DEVICE):
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram     = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "-"
    w = 58
    print("\n" + "╔" + "═" * w + "╗")
    print(f"║  {title:<{w-4}}║")
    print(f"║  Device : {gpu_name} ({device})" + " " * max(0, w - 14 - len(gpu_name) - len(str(device))) + "║")
    print(f"║  VRAM   : {vram}" + " " * max(0, w - 12 - len(vram)) + "║")
    print("╚" + "═" * w + "╝")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def print_epoch(epoch, total, loss, metric_name, metric_val, elapsed):
    print(f"  Epoch [{epoch:>2}/{total}] │ Loss: {loss:.4f} │ "
          f"{metric_name}: {metric_val:.4f} │ Time: {elapsed:.1f}s")

def print_best(model_name, best_params, metric_name, metric_val, total_time):
    print(f"\n  ✔ {model_name} — Best {metric_name}: {metric_val:.4f}")
    print(f"    Params : {best_params}")
    print(f"    Total  : {total_time:.1f}s\n")

In [ ]:
# ============================================================
# 1. LOAD, CLEAN, LABEL
# ============================================================
t0_load = time.time()

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    target_col = 'content' if 'content' in df.columns else 'Review'
    clean_df = df[[target_col]].dropna().drop_duplicates()
    clean_df.columns = ['content']
    return clean_df.reset_index(drop=True)

def clean_text_light(text: str) -> str:
    """Pembersihan ringan Bahasa Inggris: noise & emoji saja."""
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sia = SentimentIntensityAnalyzer()

def label_three_class(text, pos_thr=0.05, neg_thr=-0.05):
    """[Inference] Ambang batas VADER standar."""
    score = sia.polarity_scores(text)['compound']
    if score >= pos_thr:
        return score, 'positive'
    elif score <= neg_thr:
        return score, 'negative'
    return score, 'neutral'

df_raw = load_dataset('ulasan_aplikasi.csv')
df_raw['text_clean'] = df_raw['content'].apply(clean_text_light)

scores_labels = df_raw['content'].apply(label_three_class)
df_raw['polarity_score'] = scores_labels.apply(lambda x: x[0])
df_raw['polarity']       = scores_labels.apply(lambda x: x[1])
df_raw['label']          = df_raw['polarity'].map(LABEL_MAP)

print(f"Total data: {df_raw.shape[0]}  |  Load time: {time.time()-t0_load:.2f}s")
print(df_raw['polarity'].value_counts())

In [ ]:
# ============================================================
# 2. HELPER UMUM
# ============================================================
def make_split(df, text_col, label_col, test_ratio, val_ratio_of_train=0.1, seed=RANDOM_STATE):
    X_trainfull, X_test, y_trainfull, y_test = train_test_split(
        df[text_col], df[label_col], test_size=test_ratio,
        stratify=df[label_col], random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainfull, y_trainfull, test_size=val_ratio_of_train,
        stratify=y_trainfull, random_state=seed
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def get_class_weights(y_train_labels):
    weights = compute_class_weight('balanced',
                                   classes=np.array(list(LABEL_MAP.values())),
                                   y=y_train_labels)
    return torch.tensor(weights, dtype=torch.float)

def log_result(model_name, split_info, feature_info,
               y_train_true, y_train_pred, y_test_true, y_test_pred, best_params):
    result = {
        "Model"         : model_name,
        "Split"         : split_info,
        "Fitur"         : feature_info,
        "Best Params"   : str(best_params),
        "Train Accuracy": accuracy_score(y_train_true, y_train_pred),
        "Test Accuracy" : accuracy_score(y_test_true, y_test_pred),
        "Precision"     : precision_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "Recall"        : recall_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "F1-Score"      : f1_score(y_test_true, y_test_pred, average='macro', zero_division=0),
    }
    experiment_results.append(result)
    return result

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(LABEL_MAP.keys()),
                yticklabels=list(LABEL_MAP.keys()))
    plt.title(title); plt.xlabel('Prediksi'); plt.ylabel('Aktual')
    plt.tight_layout(); plt.show()

def build_vocab(texts, min_freq=2):
    from collections import Counter
    counter = Counter()
    for t in texts:
        counter.update(word_tokenize(t.lower()))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def text_to_ids(text, vocab, max_len):
    tokens = word_tokenize(text.lower())[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

# ── DataLoader kwargs standar untuk Colab T4 ────────────────
DL_KWARGS = dict(num_workers=2, pin_memory=True, persistent_workers=True)

In [ ]:
# ============================================================
# 3. MODEL A — TextCNN, Split 80:20
# ============================================================
torch.cuda.empty_cache()
SPLIT_TEST_RATIO_A = 0.20

print_banner("MODEL A — TextCNN  (Split 80:20)")

X_tr_A, X_val_A, X_te_A, y_tr_A, y_val_A, y_te_A = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_A
)

MAX_LEN_A    = 40
vocab_A      = build_vocab(X_tr_A, min_freq=2)
VOCAB_SIZE_A = len(vocab_A)
print(f"  Vocabulary size (train only): {VOCAB_SIZE_A}")

class TextCNNDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        # Tokenisasi SEKALI di init → jauh lebih cepat
        self.ids    = [text_to_ids(t, vocab, max_len) for t in texts]
        self.labels = labels.values if hasattr(labels, 'values') else labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        return (torch.tensor(self.ids[idx],    dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, num_filters=100,
                 kernel_sizes=(3, 4, 5), num_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x).permute(0, 2, 1)
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        pooled    = [torch.max(c, dim=2)[0] for c in conv_outs]
        concat    = torch.cat(pooled, dim=1)
        return self.fc(self.dropout(concat))

def train_textcnn(num_filters, dropout, lr, epochs=8):
    model = TextCNN(VOCAB_SIZE_A, num_filters=num_filters, dropout=dropout).to(DEVICE)
    class_weights = get_class_weights(y_tr_A.values).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=class_weights)
    optimizer     = torch.optim.Adam(model.parameters(), lr=lr)
    scaler        = GradScaler()                          # ← AMP scaler

    train_loader = DataLoader(TextCNNDataset(X_tr_A, y_tr_A, vocab_A, MAX_LEN_A),
                              batch_size=64, shuffle=True, **DL_KWARGS)   # ← batch 64
    val_loader   = DataLoader(TextCNNDataset(X_val_A, y_val_A, vocab_A, MAX_LEN_A),
                              batch_size=128, **DL_KWARGS)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)         # ← lebih cepat dari zero_grad()
            with autocast():                              # ← FP16 forward
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        # ── validasi ──
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), autocast():
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc  = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc, time.time() - t_ep)

    return model, val_acc

# --- Grid Search ---
param_grid_A = {'num_filters': [64, 100, 150], 'dropout': [0.2, 0.3], 'lr': [1e-3, 5e-4]}
best_val_acc_A, best_model_A, best_params_A = -1, None, None
t_start_A = time.time()

for num_filters, dropout, lr in itertools.product(
    param_grid_A['num_filters'], param_grid_A['dropout'], param_grid_A['lr']
):
    print(f"\n  ▸ num_filters={num_filters}, dropout={dropout}, lr={lr}")
    model, val_acc = train_textcnn(num_filters, dropout, lr, epochs=8)
    if val_acc > best_val_acc_A:
        best_val_acc_A, best_model_A = val_acc, model
        best_params_A = {'num_filters': num_filters, 'dropout': dropout, 'lr': lr}

print_best("Model A: TextCNN", best_params_A, "Val Acc", best_val_acc_A,
           time.time() - t_start_A)

def predict_dl_model(model, dataset_cls, texts, vocab=None, max_len=None, w2v=None):
    dummy_labels = pd.Series(np.zeros(len(texts)))
    if vocab is not None:
        loader = DataLoader(dataset_cls(texts, dummy_labels, vocab, max_len),
                            batch_size=128, **DL_KWARGS)
    else:
        loader = DataLoader(dataset_cls(texts, dummy_labels, w2v),
                            batch_size=128, **DL_KWARGS)
    model.eval()
    preds = []
    with torch.no_grad(), autocast():
        for xb, _ in loader:
            preds.extend(torch.argmax(model(xb.to(DEVICE, non_blocking=True)),
                                      dim=1).cpu().numpy())
    return preds

y_pred_train_A = predict_dl_model(best_model_A, TextCNNDataset, X_tr_A,
                                   vocab=vocab_A, max_len=MAX_LEN_A)
y_pred_test_A  = predict_dl_model(best_model_A, TextCNNDataset, X_te_A,
                                   vocab=vocab_A, max_len=MAX_LEN_A)

res_A = log_result("Model A: TextCNN",
                    f"{int((1-SPLIT_TEST_RATIO_A)*100)}:{int(SPLIT_TEST_RATIO_A*100)}",
                    "Trainable Embedding (dari nol)",
                    y_tr_A, y_pred_train_A, y_te_A, y_pred_test_A, best_params_A)
print(f"  Test Accuracy: {res_A['Test Accuracy']:.4f}  |  F1: {res_A['F1-Score']:.4f}")
plot_confusion(y_te_A, y_pred_test_A, "Confusion Matrix — Model A (TextCNN)")

In [ ]:
# ============================================================
# 4. MODEL B — BiLSTM + GloVe Twitter Pre-trained, Split 70:30
# ============================================================
torch.cuda.empty_cache()
SPLIT_TEST_RATIO_B = 0.30

print_banner("MODEL B — BiLSTM + GloVe Twitter  (Split 70:30)")

X_tr_B, X_val_B, X_te_B, y_tr_B, y_val_B, y_te_B = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_B
)

MAX_LEN_B     = 40
EMBED_DIM_B   = 100
GLOVE_PATH    = "glove.twitter.27B.100d.txt"

vocab_B      = build_vocab(X_tr_B, min_freq=2)
VOCAB_SIZE_B = len(vocab_B)
print(f"  Vocabulary size (train only): {VOCAB_SIZE_B}")

def load_glove_embeddings(glove_path, vocab, embed_dim):
    embedding_matrix = np.random.uniform(-0.05, 0.05,
                                         (len(vocab), embed_dim)).astype(np.float32)
    embedding_matrix[vocab['<PAD>']] = np.zeros(embed_dim)
    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word  = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f"  GloVe coverage: {found}/{len(vocab)} ({found/len(vocab)*100:.1f}%)")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

glove_embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab_B, EMBED_DIM_B)

class BiLSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN_B):
        # ★ OPTIMASI KUNCI: pre-tokenize di init, bukan di __getitem__
        self.ids    = [text_to_ids(t, vocab, max_len) for t in texts]
        self.labels = labels.values if hasattr(labels, 'values') else labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        return (torch.tensor(self.ids[idx],    dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class BiLSTMGloVe(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=64, num_layers=1,
                 num_classes=3, dropout=0.3, freeze_embed=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embed, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        emb = self.embedding(x)
        _, (hn, _) = self.lstm(emb)
        last_hidden = torch.cat((hn[-2], hn[-1]), dim=1)
        return self.fc(self.dropout(last_hidden))

def train_bilstm_glove(hidden_dim, num_layers, lr, freeze_embed, epochs=6):
    model = BiLSTMGloVe(glove_embedding_matrix, hidden_dim=hidden_dim,
                        num_layers=num_layers, freeze_embed=freeze_embed).to(DEVICE)
    class_weights = get_class_weights(y_tr_B.values).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=class_weights)
    optimizer     = torch.optim.Adam(model.parameters(), lr=lr)
    scaler        = GradScaler()                          # ← AMP

    train_loader = DataLoader(BiLSTMDataset(X_tr_B, y_tr_B, vocab_B),
                              batch_size=64, shuffle=True, **DL_KWARGS)
    val_loader   = DataLoader(BiLSTMDataset(X_val_B, y_val_B, vocab_B),
                              batch_size=128, **DL_KWARGS)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), autocast():
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc  = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc, time.time() - t_ep)

    return model, val_acc

# --- Grid Search ---
param_grid_B = {'hidden_dim': [64, 128], 'num_layers': [1, 2],
                'lr': [1e-3, 5e-4], 'freeze_embed': [True, False]}
best_val_acc_B, best_model_B, best_params_B = -1, None, None
t_start_B = time.time()

for hidden_dim, num_layers, lr, freeze_embed in itertools.product(
    param_grid_B['hidden_dim'], param_grid_B['num_layers'],
    param_grid_B['lr'], param_grid_B['freeze_embed']
):
    print(f"\n  ▸ hidden={hidden_dim}, layers={num_layers}, lr={lr}, freeze={freeze_embed}")
    model, val_acc = train_bilstm_glove(hidden_dim, num_layers, lr, freeze_embed, epochs=6)
    if val_acc > best_val_acc_B:
        best_val_acc_B, best_model_B = val_acc, model
        best_params_B = {'hidden_dim': hidden_dim, 'num_layers': num_layers,
                         'lr': lr, 'freeze_embed': freeze_embed}

print_best("Model B: BiLSTM+GloVe", best_params_B, "Val Acc", best_val_acc_B,
           time.time() - t_start_B)

y_pred_train_B = predict_dl_model(best_model_B, BiLSTMDataset, X_tr_B,
                                   vocab=vocab_B, max_len=MAX_LEN_B)
y_pred_test_B  = predict_dl_model(best_model_B, BiLSTMDataset, X_te_B,
                                   vocab=vocab_B, max_len=MAX_LEN_B)

res_B = log_result("Model B: BiLSTM + GloVe Twitter",
                    f"{int((1-SPLIT_TEST_RATIO_B)*100)}:{int(SPLIT_TEST_RATIO_B*100)}",
                    "GloVe Twitter Pre-trained (100d)",
                    y_tr_B, y_pred_train_B, y_te_B, y_pred_test_B, best_params_B)
print(f"  Test Accuracy: {res_B['Test Accuracy']:.4f}  |  F1: {res_B['F1-Score']:.4f}")
plot_confusion(y_te_B, y_pred_test_B, "Confusion Matrix — Model B (BiLSTM + GloVe)")

In [ ]:
# ============================================================
# 5. MODEL C — RoBERTa Fine-tuning, Split 75:25
# ============================================================
torch.cuda.empty_cache()
SPLIT_TEST_RATIO_C = 0.25

print_banner("MODEL C — RoBERTa Fine-tuned  (Split 75:25)")

X_tr_C, X_val_C, X_te_C, y_tr_C, y_val_C, y_te_C = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_C
)

MODEL_NAME_C = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer_C  = AutoTokenizer.from_pretrained(MODEL_NAME_C)

def tokenize_texts(texts, max_length=128):
    return tokenizer_C(list(texts), padding="max_length",
                       truncation=True, max_length=max_length)

t_tok = time.time()
train_enc_C = tokenize_texts(X_tr_C)
val_enc_C   = tokenize_texts(X_val_C)
test_enc_C  = tokenize_texts(X_te_C)
print(f"  Tokenization done in {time.time()-t_tok:.1f}s")

class HFDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels.values if hasattr(labels, 'values') else labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self): return len(self.labels)

train_dataset_C = HFDataset(train_enc_C, y_tr_C)
val_dataset_C   = HFDataset(val_enc_C,   y_val_C)
test_dataset_C  = HFDataset(test_enc_C,  y_te_C)

class_weights_C = get_class_weights(y_tr_C.values)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average='macro', zero_division=0)}

# --- Random Search: 3 kombinasi ---
param_space_C = {
    'learning_rate':              [3e-5, 2e-5, 1e-5],
    'per_device_train_batch_size': [16, 32],
    'warmup_ratio':               [0.06, 0.1]
}
sampled_combos = random.sample(list(itertools.product(
    param_space_C['learning_rate'],
    param_space_C['per_device_train_batch_size'],
    param_space_C['warmup_ratio']
)), k=3)

best_val_f1_C, best_trainer_C, best_params_C = -1, None, None
t_start_C = time.time()

for lr, batch_size, warmup in sampled_combos:
    print(f"\n  ▸ lr={lr}, bs={batch_size}, warmup={warmup}")
    t_combo = time.time()

    model_c = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME_C, num_labels=3, ignore_mismatched_sizes=True
    )
    # ★ Aktifkan gradient checkpointing → hemat VRAM, bisa pakai batch lebih besar
    model_c.gradient_checkpointing_enable()

    n_params = count_parameters(model_c)
    print(f"    Parameters: {n_params:,}")

    args_c = TrainingArguments(
        output_dir=f'./results_C_lr{lr}_bs{batch_size}',
        num_train_epochs=3,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        fp16=True,                              # ← AMP sudah ada
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=warmup,
        gradient_accumulation_steps=2,          # ← efektif batch ×2 tanpa VRAM ekstra
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=2,               # ← paralel data loading
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )
    trainer_c = WeightedTrainer(
        model=model_c, args=args_c,
        train_dataset=train_dataset_C, eval_dataset=val_dataset_C,
        compute_metrics=compute_metrics, class_weights=class_weights_C,
    )
    trainer_c.train()
    val_metrics = trainer_c.evaluate(eval_dataset=val_dataset_C)
    print(f"    Val F1-macro: {val_metrics['eval_f1_macro']:.4f}  |  "
          f"Time: {time.time()-t_combo:.1f}s")

    if val_metrics['eval_f1_macro'] > best_val_f1_C:
        best_val_f1_C  = val_metrics['eval_f1_macro']
        best_trainer_C = trainer_c
        best_params_C  = {'learning_rate': lr, 'batch_size': batch_size,
                          'warmup_ratio': warmup}

print_best("Model C: RoBERTa", best_params_C, "Val F1", best_val_f1_C,
           time.time() - t_start_C)

test_pred_C    = best_trainer_C.predict(test_dataset_C)
y_pred_test_C  = np.argmax(test_pred_C.predictions, axis=-1)
y_pred_train_C = np.argmax(best_trainer_C.predict(train_dataset_C).predictions, axis=-1)

res_C = log_result("Model C: RoBERTa Fine-tuned",
                    f"{int((1-SPLIT_TEST_RATIO_C)*100)}:{int(SPLIT_TEST_RATIO_C*100)}",
                    "Pre-trained Contextual Embeddings (Transformer)",
                    y_tr_C.values, y_pred_train_C, y_te_C.values, y_pred_test_C,
                    best_params_C)
print(f"  Test Accuracy: {res_C['Test Accuracy']:.4f}  |  F1: {res_C['F1-Score']:.4f}")
plot_confusion(y_te_C.values, y_pred_test_C, "Confusion Matrix — Model C (RoBERTa)")

In [ ]:
# ============================================================
# 6. TABEL PERBANDINGAN 3 MODEL
# ============================================================
torch.cuda.empty_cache()

df_comparison = pd.DataFrame(experiment_results).sort_values("Test Accuracy", ascending=False)
print("\n╔══════════════════════════════════════════════════════════╗")
print("║        PERBANDINGAN 3 MODEL DEEP LEARNING               ║")
print("╚══════════════════════════════════════════════════════════╝")
print(df_comparison.to_string(index=False))

best_model_name = df_comparison.iloc[0]["Model"]
print(f"\n  🏆 Model terbaik (Test Accuracy): {best_model_name}\n")

fig, ax = plt.subplots(figsize=(9, 5))
df_comparison.set_index("Model")[["Train Accuracy", "Test Accuracy"]].plot(
    kind='bar', ax=ax, color=['#4C72B0', '#DD8452']
)
ax.set_title("Perbandingan Train vs Test Accuracy — 3 Model Deep Learning")
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

def plot_wordcloud_by_class(df, text_col, label_col, label_value, color_map):
    text_subset = ' '.join(df[df[label_col] == label_value][text_col])
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=color_map).generate(text_subset)
    plt.figure(figsize=(8, 4))
    plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
    plt.title(f"WordCloud — Kelas '{label_value}'")
    plt.show()

for label_value, cmap in [('positive','Greens'), ('neutral','Greys'), ('negative','Reds')]:
    plot_wordcloud_by_class(df_raw, 'text_clean', 'polarity', label_value, cmap)

# ============================================================
# 7. INFERENCE — contoh menggunakan Model A
# ============================================================
def predict_textcnn(text_review, model, vocab, max_len):
    cleaned = clean_text_light(text_review)
    ids = text_to_ids(cleaned, vocab, max_len)
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    model.eval()
    with torch.no_grad(), autocast():
        logits = model(x)
        probs  = torch.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return ID2LABEL[pred_idx], float(probs[pred_idx])

test_reviews = [
    "The storyline is super amazing and the English voice acting is top-tier!",
    "Too many bugs after the update, the game keeps crashing on the loading screen.",
    "The game is okay, average gacha mechanics."
]

print("\n=== HASIL INFERENCE (Model A: TextCNN) ===")
for review in test_reviews:
    sentimen, conf = predict_textcnn(review, best_model_A, vocab_A, MAX_LEN_A)
    print(f'Ulasan   : "{review}"')
    print(f"Sentimen : {sentimen.upper()} (Confidence: {conf*100:.2f}%)")
    print("-" * 50)

In [ ]:
# ============================================================
# 7. INFERENCE — contoh menggunakan Model A (ganti sesuai model terbaik Anda)
# ============================================================
def predict_textcnn(text_review, model, vocab, max_len):
    cleaned = clean_text_light(text_review)
    ids = text_to_ids(cleaned, vocab, max_len)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return ID2LABEL[pred_idx], float(probs[pred_idx])

test_reviews = [
    "The storyline is super amazing and the English voice acting is top-tier!",
    "Too many bugs after the update, the game keeps crashing on the loading screen.",
    "The game is okay, average gacha mechanics."
]

print("=== HASIL INFERENCE (Model A: TextCNN) ===")
for review in test_reviews:
    sentimen, conf = predict_textcnn(review, best_model_A, vocab_A, MAX_LEN_A)
    print(f"Ulasan   : \"{review}\"")
    print(f"Sentimen : {sentimen.upper()} (Confidence: {conf*100:.2f}%)")
    print("-" * 50)